In [ ]:
import json
from pathlib import Path

base = Path.home() / "POPE/output/coco"
out_dir = Path.home() / "learn-to-steer/data/pope_descriptive"
out_dir.mkdir(parents=True, exist_ok=True)


files = {
    "random": base / "coco_pope_random.json",
    "popular": base / "coco_pope_popular.json",
    "adversarial": base / "coco_pope_adversarial.json",
}

merged = []

for subset, path in files.items():
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            merged.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

out_path = out_dir / "annotations.json"
with open(out_path, "w") as f:
    json.dump(merged, f, indent=2)

print(f"Saved {len(merged)} examples to {out_path}")

Saved 9000 examples to /research/hal-afsharim/learn-to-steer/data/pope/train/annotations.json


In [1]:
import json
import random
from pathlib import Path

import numpy as np
import torch

def set_seed(seed_value=0):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    
seed = 0
set_seed(seed)

# Change these paths if needed
pope_root = Path("/research/hal-afsharim/POPE/output/coco")
out_root = Path("/research/hal-afsharim/POPE/pope_3example_20000")
coco_val2014 = Path("/research/hal-afsharim/learn-to-steer/data/coco/val2014")

files = {
    "random": pope_root / "coco_pope_random.json",
    "popular": pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

splits = {
    "train": [],
    "val": [],
    "test": [],
}

for subset, path in files.items():
    subset_data = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)  # POPE files are JSONL
            subset_data.append({
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            })

    random.shuffle(subset_data)

    n = len(subset_data)
    n_train = int(0.7 * n)
    n_val = int(0.1 * n)
    n_test = n - n_train - n_val

    splits["train"].extend(subset_data[:n_train])
    splits["val"].extend(subset_data[n_train:n_train + n_val])
    splits["test"].extend(subset_data[n_train + n_val:])

# Optional: shuffle within each final split so subsets are mixed
for split_name in splits:
    random.shuffle(splits[split_name])

for split_name, split_data in splits.items():
    split_dir = out_root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    with open(split_dir / "annotations.json", "w") as f:
        json.dump(split_data, f, indent=2)

    images_link = split_dir / "images"
    if images_link.is_symlink() or images_link.exists():
        images_link.unlink()
    images_link.symlink_to(coco_val2014)

    print(f"{split_name}: {len(split_data)} samples -> {split_dir / 'annotations.json'}")

print("Per-split subset counts:")
for split_name, split_data in splits.items():
    counts = {}
    for x in split_data:
        counts[x["subset"]] = counts.get(x["subset"], 0) + 1
    print(split_name, counts)

print(f"Total: {sum(len(v) for v in splits.values())}")

train: 252000 samples -> /research/hal-afsharim/POPE/pope_3example_20000/train/annotations.json
val: 36000 samples -> /research/hal-afsharim/POPE/pope_3example_20000/val/annotations.json
test: 72000 samples -> /research/hal-afsharim/POPE/pope_3example_20000/test/annotations.json
Per-split subset counts:
train {'popular': 84000, 'random': 84000, 'adversarial': 84000}
val {'popular': 12000, 'adversarial': 12000, 'random': 12000}
test {'adversarial': 24000, 'random': 24000, 'popular': 24000}
Total: 360000


In [24]:
import json
import random
from pathlib import Path

import numpy as np
import torch

def set_seed(seed_value=0):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed = 0
set_seed(seed)

# Change these paths if needed
pope_root = Path("/research/hal-afsharim/POPE/output/coco")
out_root = Path("/research/hal-afsharim/POPE/pope_3example_5000")
coco_val2014 = Path("/research/hal-afsharim/learn-to-steer/data/coco/val2014")

files = {
    "random": pope_root / "coco_pope_random.json",
    "popular": pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

# Collect all examples, grouped by image so each image stays in one split
by_image = {}
for subset, path in files.items():
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)  # POPE files are JSONL
            item = {
                "filename": ex["image"],
                "instruction": ex["text"] + " Answer with just one word.",
                "response": ex["label"],
                "subset": subset,
            }
            by_image.setdefault(ex["image"], []).append(item)

# Split by image id to avoid overlap across train/val/test
image_ids = list(by_image.keys())
random.shuffle(image_ids)

n_images = len(image_ids)
n_train = int(0.7 * n_images)
n_val = int(0.1 * n_images)
n_test = n_images - n_train - n_val

train_images = set(image_ids[:n_train])
val_images = set(image_ids[n_train:n_train + n_val])
test_images = set(image_ids[n_train + n_val:])

splits = {
    "train": [],
    "val": [],
    "test": [],
}

for image_id, items in by_image.items():
    if image_id in train_images:
        splits["train"].extend(items)
    elif image_id in val_images:
        splits["val"].extend(items)
    else:
        splits["test"].extend(items)

# Optional: shuffle within each final split so subsets are mixed
for split_name in splits:
    random.shuffle(splits[split_name])

for split_name, split_data in splits.items():
    split_dir = out_root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    with open(split_dir / "annotations.json", "w") as f:
        json.dump(split_data, f, indent=2)

    images_link = split_dir / "images"
    if images_link.is_symlink() or images_link.exists():
        images_link.unlink()
    images_link.symlink_to(coco_val2014)

    print(f"{split_name}: {len(split_data)} samples -> {split_dir / 'annotations.json'}")

print("Per-split subset counts:")
for split_name, split_data in splits.items():
    counts = {}
    for x in split_data:
        counts[x["subset"]] = counts.get(x["subset"], 0) + 1
    print(split_name, counts)

print(f"Total: {sum(len(v) for v in splits.values())}")

train: 47406 samples -> /research/hal-afsharim/POPE/pope_3example_5000/train/annotations.json
val: 6768 samples -> /research/hal-afsharim/POPE/pope_3example_5000/val/annotations.json
test: 13554 samples -> /research/hal-afsharim/POPE/pope_3example_5000/test/annotations.json
Per-split subset counts:
train {'random': 15802, 'adversarial': 15802, 'popular': 15802}
val {'popular': 2256, 'random': 2256, 'adversarial': 2256}
test {'random': 4518, 'adversarial': 4518, 'popular': 4518}
Total: 67728


In [ ]:
# # Collect all examples, grouped by image so each image stays in one split.
# # Dedupe by (image, question) because the random/popular/adversarial files
# # reuse the same positives and overlapping negatives for the same images.
# by_image = {}
# seen = set()                      # (image, instruction) already added
# for subset, path in files.items():
#     with open(path, "r") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             ex = json.loads(line)  # POPE files are JSONL
#             instruction = ex["text"] + " Answer with just one word."
#             key = (ex["image"], instruction)
#             if key in seen:        # same question already collected
#                 continue
#             seen.add(key)
#             item = {
#                 "filename": ex["image"],
#                 "instruction": instruction,
#                 "response": ex["label"],
#                 "subset": subset,  # first subset that contributed this question
#             }
#             by_image.setdefault(ex["image"], []).append(item)


In [18]:
import json

with open(out_root / "train" / "annotations.json", "r") as f:
    train_data = json.load(f)
print(f"Train set has {len(train_data)} examples")
file_names_train = [ex["filename"] for ex in train_data]
print(f"Unique images in train set: {len(set(file_names_train))}")

with open(out_root / "val" / "annotations.json", "r") as f:
    val_data = json.load(f)
print(f"Val set has {len(val_data)} examples")
file_names_val = [ex["filename"] for ex in val_data]
print(f"Unique images in val set: {len(set(file_names_val))}")

with open(out_root / "test" / "annotations.json", "r") as f:
    test_data = json.load(f)
print(f"Test set has {len(test_data)} examples")
file_names_test = [ex["filename"] for ex in test_data]
print(f"Unique images in test set: {len(set(file_names_test))}")

Train set has 42000 examples
Unique images in train set: 7000
Val set has 6000 examples
Unique images in val set: 1000
Test set has 12000 examples
Unique images in test set: 2000


In [19]:
#check for overlap
overlap_train_val = set(file_names_train) & set(file_names_val)
overlap_train_test = set(file_names_train) & set(file_names_test)
overlap_val_test = set(file_names_val) & set(file_names_test)
print(f"Overlap between train and val: {len(overlap_train_val)} images")
print(f"Overlap between train and test: {len(overlap_train_test)} images")
print(f"Overlap between val and test: {len(overlap_val_test)} images")

Overlap between train and val: 0 images
Overlap between train and test: 0 images
Overlap between val and test: 0 images


In [22]:
import json, itertools, collections
from pathlib import Path

pope_root = Path("/research/hal-afsharim/POPE/output/coco")
files = {
    "random":      pope_root / "coco_pope_random.json",
    "popular":     pope_root / "coco_pope_popular.json",
    "adversarial": pope_root / "coco_pope_adversarial.json",
}

# Load each subset as a set of (image, question, label)
subsets = {}
for name, path in files.items():
    rows = set()
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            rows.add((ex["image"], ex["text"], ex["label"]))
    subsets[name] = rows
    print(f"{name:12s}: {len(rows):6d} rows")

print("\n--- Pairwise overlap (same image+question) ---")
for a, b in itertools.combinations(subsets, 2):
    # overlap on (image, question) regardless of label
    qa = {(i, t) for i, t, l in subsets[a]}
    qb = {(i, t) for i, t, l in subsets[b]}
    inter = qa & qb
    print(f"{a:12s} ∩ {b:12s}: {len(inter):6d} shared questions")

print("\n--- Shared across all three ---")
common = set.intersection(*[{(i, t) for i, t, l in s} for s in subsets.values()])
print(f"in all 3 subsets: {len(common)} questions")

print("\n--- Duplicates in the merged set ---")
merged = [(i, t) for s in subsets.values() for (i, t, l) in s]
counter = collections.Counter(merged)
dups = {k: v for k, v in counter.items() if v > 1}
print(f"total merged rows : {len(merged)}")
print(f"distinct questions: {len(counter)}")
print(f"redundant rows    : {len(merged) - len(counter)}")
print(f"multiplicity dist : {dict(collections.Counter(dups.values()))}")

print("\n--- Examples of overlapping questions ---")
for (img, txt), n in list(dups.items())[:5]:
    in_subsets = [name for name, s in subsets.items()
                  if any(i == img and t == txt for i, t, l in s)]
    print(f"  {img}  |  {txt}  ->  in {in_subsets}")

# Sanity check: does any shared question have conflicting labels?
labels = collections.defaultdict(set)
for s in subsets.values():
    for i, t, l in s:
        labels[(i, t)].add(l)
conflicts = {k: v for k, v in labels.items() if len(v) > 1}
print(f"\nconflicting labels: {len(conflicts)} (should be 0)")


random      :  22450 rows
popular     :  22450 rows
adversarial :  22450 rows

--- Pairwise overlap (same image+question) ---
random       ∩ popular     :      0 shared questions
random       ∩ adversarial :      0 shared questions
popular      ∩ adversarial :      0 shared questions

--- Shared across all three ---
in all 3 subsets: 0 questions

--- Duplicates in the merged set ---
total merged rows : 67350
distinct questions: 67350
redundant rows    : 0
multiplicity dist : {}

--- Examples of overlapping questions ---

conflicting labels: 0 (should be 0)
